# Line Bets: Pass Line and Don't Pass

The two "line" bets are the canonical first wagers in craps, and the
first bet-level derivations in `craps-lab`. They are near mirror
images of each other, with one rule asymmetry — the **"bar 12"**
convention on don't pass — that produces small but real differences
in house edge.

This notebook derives both edges in closed form, cross-checks the
derivations against the exact `Fraction` values exposed by
`craps_lab.probability`, and validates them with a Monte Carlo
over the seeded `DiceRoller` from `craps_lab.play`.

## Pass line

Pass line resolves in two phases.

**Come-out roll**

| come-out sum | result | probability |
|:---:|:---:|:---:|
| 7, 11 | immediate win | $8/36$ |
| 2, 3, 12 | immediate loss | $4/36$ |
| 4, 5, 6, 8, 9, 10 | establish point | $24/36$ |

**Point phase.** Given an established point $p$, the shooter rolls
until either $p$ (win) or 7 (lose). Non-resolving rolls just restart
the geometric process, so the conditional probability is

$$
P(p \text{ before } 7) \;=\; \frac{\mathrm{count}(p)}{\mathrm{count}(p) + \mathrm{count}(7)} \;=\; \frac{\mathrm{count}(p)}{\mathrm{count}(p) + 6}.
$$

For the six possible points:

| point $p$ | $\mathrm{count}(p)$ | $P(p \text{ before } 7)$ |
|:---:|:---:|:---:|
| 4, 10 | 3 | $1/3$ |
| 5, 9 | 4 | $2/5$ |
| 6, 8 | 5 | $5/11$ |

Combining both phases,

$$
P(\text{pass win}) \;=\; \sum_{s \in \{7, 11\}} P(S=s) \;+\; \sum_{p \in \{4,5,6,8,9,10\}} P(S=p) \cdot P(p \text{ before } 7) \;=\; \frac{244}{495} \;\approx\; 0.4929.
$$

Pass line has no push outcomes, so

$$
\text{house edge} \;=\; P(\text{lose}) - P(\text{win}) \;=\; 1 - 2 \cdot \frac{244}{495} \;=\; \frac{7}{495} \;\approx\; 0.01414 \;=\; 1.414\%.
$$

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np

from craps_lab.bets import Outcome
from craps_lab.dice import DiceRoller
from craps_lab.play import play_dont_pass, play_pass_line
from craps_lab.probability import (
    dont_pass_house_edge,
    dont_pass_push_probability,
    dont_pass_win_probability,
    pass_line_house_edge,
    pass_line_win_probability,
)

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 0xB0BA

In [ ]:
pass_win = pass_line_win_probability()
pass_lose = Fraction(1) - pass_win
pass_edge = pass_line_house_edge()

print(f"P(pass win)  = {pass_win}  = {float(pass_win):.6f}")
print(f"P(pass lose) = {pass_lose}  = {float(pass_lose):.6f}")
print(f"house edge   = {pass_edge}    = {float(pass_edge):.6f}  ({100 * float(pass_edge):.4f}%)")

## Don't pass

Don't pass is the inverse of pass line with one crucial rule change:
the **bar-12** convention turns the come-out 12 into a *push* rather
than a win. Without it, don't pass would have a slight player edge;
the bar rule is what preserves the house advantage.

**Come-out roll**

| come-out sum | result | probability |
|:---:|:---:|:---:|
| 2, 3 | immediate win | $3/36$ |
| 12 | **push (bar 12)** | $1/36$ |
| 7, 11 | immediate loss | $8/36$ |
| 4, 5, 6, 8, 9, 10 | establish point | $24/36$ |

**Point phase.** Don't pass wins if 7 comes before the point, so

$$
P(\text{win} \mid \text{point } p) \;=\; 1 - P(p \text{ before } 7).
$$

Combining,

$$
P(\text{don't win}) \;=\; P(S \in \{2, 3\}) \;+\; \sum_{p} P(S=p) \cdot \bigl(1 - P(p \text{ before } 7)\bigr) \;=\; \frac{949}{1980}.
$$

$P(\text{push}) = 1/36$, and the house edge is

$$
\text{edge} \;=\; 1 - 2 \cdot P(\text{win}) - P(\text{push}) \;=\; \frac{3}{220} \;\approx\; 0.01364 \;=\; 1.364\%,
$$

slightly less than the pass line's $1.414\%$. The reduction comes
from the point phase: don't pass resolves against the 7 (count 6)
rather than against the specific point (count 3, 4, or 5), and that
asymmetry more than offsets the house edge lost to the 12 push on
the come-out.

In [ ]:
dont_win = dont_pass_win_probability()
dont_push = dont_pass_push_probability()
dont_lose = Fraction(1) - dont_win - dont_push
dont_edge = dont_pass_house_edge()

print(f"P(dont win)  = {dont_win}  = {float(dont_win):.6f}")
print(f"P(dont push) = {dont_push}     = {float(dont_push):.6f}")
print(f"P(dont lose) = {dont_lose}  = {float(dont_lose):.6f}")
print(f"house edge   = {dont_edge}    = {float(dont_edge):.6f}  ({100 * float(dont_edge):.4f}%)")

# Cross-check the global identity:
# pass line wins iff don't pass loses (and don't pushes on 12 separately),
# so P(pass win) + P(dont win) + P(dont push) must equal 1 exactly.
identity = pass_win + dont_win + dont_push
print(f"\nP(pass win) + P(dont win) + P(dont push) = {identity}")
assert identity == Fraction(1), "global identity failed"

In [ ]:
labels = ["win", "push", "lose"]
pass_values = [float(pass_win), 0.0, float(pass_lose)]
dont_values = [float(dont_win), float(dont_push), float(dont_lose)]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width / 2, pass_values, width, color="steelblue", edgecolor="white", label="pass line")
ax.bar(x + width / 2, dont_values, width, color="tomato", edgecolor="white", label="don't pass")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("probability")
ax.set_title("Line bet resolution probabilities")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Pass line house edge:  {100 * float(pass_edge):.4f}%")
print(f"Don't pass house edge: {100 * float(dont_edge):.4f}%")
print(f"Difference (pass - dont): {100 * float(pass_edge - dont_edge):.4f}%")

## Monte Carlo convergence

Now roll the seeded `DiceRoller` through many games of each bet and
watch the running empirical edge converge on the analytical value.
The running estimate at game $n$ is

$$
\widehat{\text{edge}}_n \;=\; \frac{\#\text{losses}_n - \#\text{wins}_n}{n},
$$

with pushes contributing 0 to the sum. Its standard error is bounded
by $1/\sqrt{n}$, so the natural $\pm 2\sigma$ band shrinks as
$n^{-1/2}$.

In [ ]:
MAX_GAMES = 100_000


def running_edge(play_func, seed):
    roller = DiceRoller(seed=seed)
    trace = np.zeros(MAX_GAMES)
    lose = 0
    win = 0
    for i in range(MAX_GAMES):
        outcome = play_func(roller)
        if outcome == Outcome.WIN:
            win += 1
        elif outcome == Outcome.LOSE:
            lose += 1
        trace[i] = (lose - win) / (i + 1)
    return trace


pass_trace = running_edge(play_pass_line, SEED)
dont_trace = running_edge(play_dont_pass, SEED + 1)

n_axis = np.arange(1, MAX_GAMES + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_axis, pass_trace, color="steelblue", linewidth=1, label="pass line empirical")
ax.axhline(
    float(pass_edge),
    color="steelblue",
    linestyle="--",
    linewidth=1.5,
    label=f"pass line analytical = 7/495 ≈ {float(pass_edge):.4f}",
)
ax.plot(n_axis, dont_trace, color="tomato", linewidth=1, label="don't pass empirical")
ax.axhline(
    float(dont_edge),
    color="tomato",
    linestyle="--",
    linewidth=1.5,
    label=f"don't pass analytical = 3/220 ≈ {float(dont_edge):.4f}",
)
ax.axhline(0.0, color="black", linewidth=0.5, alpha=0.3)
ax.set_xscale("log")
ax.set_xlabel("games played")
ax.set_ylabel("empirical house edge")
ax.set_title("Running empirical edge vs. analytical house edge")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## Takeaways

1. **Pass line house edge is $7/495 \approx 1.414\%$** in exact
   closed form — a `Fraction`, no floating-point error anywhere in
   the pipeline.
2. **Don't pass house edge is $3/220 \approx 1.364\%$**, slightly
   lower thanks to the point-phase asymmetry. The edge reduction
   from resolving against the 7 instead of the point more than
   compensates for the edge lost to the bar-12 push.
3. **The two derivations are mutually consistent.** The identity
   $P(\text{pass win}) + P(\text{don't win}) + P(\text{don't push}) = 1$
   holds *exactly* (not just numerically) because pass line wins iff
   don't pass loses, and don't pass pushes separately on come-out 12.
4. **The Monte Carlo converges cleanly** on both analytical edges
   within 100,000 games on the seeded `DiceRoller`. The unit test
   in `tests/test_convergence.py` pins this down with a 5-sigma
   tolerance against the exact `Fraction` values.

**Coming next:** come and don't-come bets (same resolution logic
placed after the come-out, so a different probability-mass mix) and
**free odds** — the zero-edge side bet that dilutes a pass line's
1.414% to under 0.6% at max odds.